⚠️ **Under construction.** <!-- banner:under-construction --> This lab is drafted but not yet instructor-reviewed; it may change without notice until its session.

# Unit 5 lab · Dictionaries, `and`, and sums of logs

**Today:** by the end of the unit you can read chapter 5's code line by line. That means its tables, which are dictionaries looked up by class name, its `and`, which joins two yes-or-no questions, and its log sums, which take the place of a product of many probabilities.

**Before you start:** chapter 5 read, at least through *The story, as code* for part 1 and through the *Implementation note* on log space for part 2. This notebook comes from the unit's lab repository. At the unit's first session, run `gh student accept bu-bioinfo-classrooms bf550-fall-2026 bf550-unit05-labs`, clone what it creates, and open this notebook with the `bf550` kernel.

**One notebook for the whole unit.** Part 1 is for the unit's first session, part 2 for the second, and part 3 for the third. If you do not finish a part, carry on with it after class or at the start of the next session, in this same notebook. **At the start of every session, run the setup cell below first.** Each part needs only that cell, so you can start a part without having finished the one before it.

Each section names one idea, explains what it does, and asks you to **predict
what a cell prints before you run it**. Write the prediction down, on paper, out
loud, or in a comment. A prediction you can compare against the output is what
tells you which parts of the code you can already read.

Most sections end with a **Test your understanding** task: write a small piece
of code, then run the check cell under it. Every task has a hint in the **Hints**
block at the end of the notebook, for when you want it. The check never grades and never
breaks anything. A ⬜ means not attempted yet, a ❌ means not yet and comes with
a hint, and a ✅ means passing. Run the check cells rather than editing them.
Everything else in the notebook is yours to change.

**AI in this lab.** Until your prediction is written down, work at level 1, with
no AI. The prediction is how you find out what you can read unaided, and both
exams are level 1. Once you have run a cell, level 3 is encouraged: ask your
tutor to explain anything you missed.

Run every cell, and change things to see what happens. Nothing in this notebook
can be broken in a way that matters.

## Setup

Run this cell at the start of every session. It imports NumPy, loads the self-check, and rebuilds chapter 5's two tables of presence probabilities exactly as the chapter builds them, with seed 61.

In [ ]:
import numpy as np
from labcheck import check

# chapter 5's settings: one presence probability per panel k-mer, for each class
rng = np.random.default_rng(61)
presence_prob = {"rRNA":     rng.uniform(0.15, 0.85, size=20),
                 "non-rRNA": rng.uniform(0.15, 0.85, size=20)}
print("setup done")

## Part 1 · Values stored under names

**In this part:** you can read a dictionary, look up a value by its key one level deep or two, count its entries with `len`, and fill a dictionary by counting.

### 1.1 · A dictionary stores values under keys

A list stores values at **positions**, so `draws[3]` is the value at position 3. A **dictionary** stores values under **keys**, which are names you choose. Curly braces build one, with a colon between each key and its value. Square brackets look a value up, as in list indexing, with a key in place of the position. Chapter 5 keeps its two tables this way.

The cell below stores the chapter's two class shares under the class names. **Predict all four printed lines before you run it.**

In [ ]:
share = {"rRNA": 0.15, "non-rRNA": 0.85}
print(share["non-rRNA"])
print(len(share))
share["unlabeled"] = 0.0
print(len(share))
print(share)

The lines are 0.85, then 2, then 3, then the whole dictionary with three keys. `len` counts a dictionary's entries, one per key, as it counts a list's elements. Assigning a value to a key the dictionary does not have yet adds that key. Looking up a key it does not have stops the program with a `KeyError`. Try `share["unknown"]` in a new cell to see the message.

**Test your understanding.** Create two variables from the dictionary `reads_by_class` in the cell below, which holds how many of chapter 5's 100,000 simulated reads fell in each class. `total_reads` stores one whole number, the sum of the dictionary's values, computed with a `for` loop over its keys. `rrna_share` stores one number, the rRNA count divided by `total_reads`, rounded to 4 decimals. A `for` loop over a dictionary visits its keys, so the value inside the loop is `reads_by_class[key]`. This is question 1. Its hint is at the end of the notebook.

In [ ]:
# your turn: two variables named total_reads and rrna_share
reads_by_class = {"rRNA": 14974, "non-rRNA": 85026}

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("total_reads", expect=100000,
      hint="start at 0, then add reads_by_class[key] once for each key")
check("rrna_share", expect=0.1497,
      hint="reads_by_class['rRNA'] / total_reads, rounded to 4 decimals; chapter 5 prints the same share")

### 1.2 · Two lookups in a row

Chapter 5's `presence_prob`, rebuilt by the setup cell, is a dictionary with two keys, `"rRNA"` and `"non-rRNA"`. Under each key it stores an array of twenty presence probabilities, one per panel k-mer. One probability therefore takes two lookups, read left to right. The key picks the class's table, and the position picks the k-mer in that table. **Predict before you run: what does `len(presence_prob)` print, and what does `len(presence_prob["rRNA"])` print? Is the probability on the third line between 0.15 and 0.85?**

In [ ]:
print(len(presence_prob))
print(len(presence_prob["rRNA"]))
print(round(presence_prob["rRNA"][0], 3))
table = presence_prob["non-rRNA"]
print(np.round(table[:4], 2))

The first two lines are 2 and 20. The dictionary has two keys, and the array under one key has twenty entries, because `len` counts only the level it is given. The third line is 0.412, the presence probability of k-mer 0 among rRNA reads. It lies between 0.15 and 0.85 because the chapter drew every probability in that range. The last two lines show a readable habit. Name the inner table first (`table = presence_prob["non-rRNA"]`), then look up inside the name, which records which class you are in.

The same two-step lookup works when the inner level is itself a dictionary, called a **dictionary of dictionaries**. The cell below types presence probabilities by hand for two of the 6-mers in chapter 5's 14-base read, stored under the 6-mers' own sequences. **Predict both printed values.**

In [ ]:
presence = {"rRNA":     {"GCGTAC": 0.8, "GTTAGC": 0.3},
            "non-rRNA": {"GCGTAC": 0.2, "GTTAGC": 0.6}}
print(presence["non-rRNA"]["GCGTAC"])
table = presence["rRNA"]
print(table["GTTAGC"])

The values are 0.2 and 0.3. The first line does both lookups in one expression. The second names the inner dictionary and then looks up in it, which is easier to read aloud.

**Test your understanding.** Write a function named `class_average` that takes two parameters: `presence`, a dictionary of dictionaries shaped like the one above, and `class_name`, one of its outer keys. It returns one number, the mean of all the values stored under that class, rounded to 2 decimals. Loop over the inner dictionary's keys, and use `len` for the denominator. Then create a variable named `gap` that stores `class_average(presence, "rRNA")` minus `class_average(presence, "non-rRNA")`, rounded to 2 decimals, using the `presence` dictionary above. This is question 2. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named class_average(presence, class_name), then a variable named gap

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("class_average", expect=0.55, args=({"x": {"a": 0.5, "b": 0.6}}, "x"))
check("gap", expect=0.15,
      hint="(0.8 + 0.3) / 2 minus (0.2 + 0.6) / 2: the two classes differ by 0.15")

### 1.3 · Counting into a dictionary

A dictionary can also keep counts. Start every key at 0, then add 1 under a key each time you meet it. Here the keys are the two class names, and the values count reads. **Predict the printed dictionary.**

In [ ]:
labels = ["non-rRNA", "rRNA", "non-rRNA", "non-rRNA", "rRNA", "non-rRNA"]
tally = {"rRNA": 0, "non-rRNA": 0}
for label in labels:
    tally[label] += 1
print(tally)

The dictionary is `{'rRNA': 2, 'non-rRNA': 4}`. On each pass, `label` holds one class name, and `tally[label] += 1` adds 1 to the count stored under it. The keys had to exist first. Without the line that starts both counts at 0, the first `+= 1` stops the program with a `KeyError`, because there is no count to add to yet.

**Test your understanding.** Write a function named `count_labels` that takes one parameter, `labels`, a list of class names, each `"rRNA"` or `"non-rRNA"`. It returns a dictionary with those two keys, each storing how many times its name appears in `labels`. A class that never appears still gets its key, with the count 0. This is question 3. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named count_labels(labels)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("count_labels", expect={"rRNA": 1, "non-rRNA": 2},
      args=(["rRNA", "non-rRNA", "non-rRNA"],))
check("count_labels", expect={"rRNA": 0, "non-rRNA": 0}, args=([],),
      hint="start both keys at 0 before the loop; an empty list then needs no extra code")

### If you finish part 1 early

- Chapter 5's `draw_labeled_read` looks up `presence_prob[label]`, with the key held in a variable. Find that line in *One read, drawn by a function* and read it aloud the way the chapter's box does.
- Type a list of thirty class names, predict what `count_labels` returns for it, then run it.
- PS5 is open.

### If you are stuck in part 1

Wave someone over. This hour exists so that a stuck step costs you a minute
rather than an evening. Known snags:

- **`KeyError`.** You looked up a key the dictionary does not have. Print the dictionary and compare its keys with the one you asked for. Capital letters and hyphens count, so `"rRNA"` and `"rrna"` are different keys.
- **`total_reads` holds a class name.** A loop over a dictionary gives its keys. The value stored under a key is `reads_by_class[key]`.
- **`count_labels([])` is missing a key.** Start both counts at 0 before the loop.

## Part 2 · `and`, and products of many probabilities

**In this part:** you can join two yes-or-no questions with `and` or `or`, and you can explain why chapter 5 adds logarithms where it would otherwise multiply thousands of probabilities.

### 2.1 · `and` and `or`

Two yes-or-no questions join with plain words. `and` is True only when both are True. `or` is True when at least one of them is True. Chapter 5 counts the reads that have two k-mers with `if read[0] and read[1]:`. The cell below asks three questions of one read's first four k-mer answers, typed by hand. **Predict which lines print.**

In [ ]:
read = [True, False, True, True]    # four k-mer answers: True where the k-mer is present
if read[0] and read[2]:
    print("k-mers 0 and 2 are both present")
if read[1] or read[3]:
    print("k-mer 1 or k-mer 3 is present")
if read[0] and read[1]:
    print("k-mers 0 and 1 are both present")

The first two lines print, and the third does not, because k-mer 1 is absent. Python also stops early in an `and`. If the first question is already False, it does not evaluate the second.

**Test your understanding.** Write a function named `is_false_discard` that takes two parameters: `posterior`, one number, the posterior probability that a read is rRNA; and `truth`, the read's true class, `"rRNA"` or `"non-rRNA"`. It returns True when the read is discarded **and** is a non-rRNA read, and False otherwise. Chapter 5 discards a read when its posterior is at least 0.9. Write the rule as two comparisons joined by one `and`. This is question 4. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named is_false_discard(posterior, truth)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("is_false_discard", expect=True, args=(0.95, "non-rRNA"))
check("is_false_discard", expect=False, args=(0.95, "rRNA"))
check("is_false_discard", expect=False, args=(0.89, "non-rRNA"),
      hint="at least 0.9: is 0.89 in?")
check("is_false_discard", expect=True, args=(0.9, "non-rRNA"),
      hint="at least 0.9 includes 0.9 itself, so the comparison is >= rather than >")

### 2.2 · The product that reaches zero

Multiply many probabilities, each below 1, and the product falls toward zero. Below the smallest number the computer can store, it stores exactly 0.0, and no error message says so. Chapter 5 calls this **underflow**. **Predict: which of the three products below print as a number above zero?**

In [ ]:
for n_factors in (100, 500, 2000):
    product = 1.0
    for _ in range(n_factors):
        product = product * 0.3
    print(n_factors, "factors:", product)

A hundred factors give about 5e-53, which is tiny and still stored. The notation is scientific, as in chapter 5, so `5e-53` means 5 times ten to the power of minus 53. Five hundred factors give about 4e-262, close to the limit. Two thousand give exactly **0.0**. Any code that multiplies enough probabilities reaches 0.0 at some number of factors.

**Test your understanding.** Write a function named `steps_to_zero` that takes one parameter, `factor`, a number between 0 and 1. It multiplies `factor` into a running product that starts at 1.0, one multiplication at a time. It returns one whole number, how many multiplications it took for the product to become exactly 0.0. Use a `for` loop over `range(1, 100001)` with a `return` inside it, so that the function stops at the first step where the product equals 0.0. This is question 5. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named steps_to_zero(factor)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("steps_to_zero", expect=619, args=(0.3,),
      hint="product = product * 0.3 on each pass; return the step number as soon as product == 0.0")
check("steps_to_zero", expect=380, args=(0.14,))

### 2.3 · Logs turn the product into a sum

`np.log(x)` is the logarithm of `x`. One property does all the work: **log(a × b) = log(a) + log(b)**. A product of 2,000 factors therefore becomes a sum of 2,000 logarithms. Each logarithm is a moderate negative number, and their sum stays far from any limit. Logarithms also keep order, so the larger of two numbers has the larger logarithm. `np.log(0.3)` is about −1.2. **Predict roughly: what does the sum of 2,000 copies of `np.log(0.3)` print?**

In [ ]:
log_total = 0.0
for _ in range(2000):
    log_total = log_total + np.log(0.3)
print(round(log_total, 1))

The sum is −2407.9, which is 2,000 times −1.204. For the same 2,000 factors, the product in section 2.2 printed 0.0, and the sum is an ordinary number. Chapter 5's `log_likelihood` does the same for the classifier and compares the two classes' log sums in place of their products.

**Test your understanding.** Chapter 5's *Implementation note* scores one read against 2,000 k-mers and prints two log sums, −1206.8 under rRNA and −1612.5 under non-rRNA. Create two variables from `score_rrna` and `score_nonrrna` in the cell below. `winner` stores one string, `"rRNA"` or `"non-rRNA"`, the class with the larger score, chosen in code with an `if`/`else` and a comparison. `log_gap` stores one positive number, the winner's score minus the other score, rounded to 1 decimal. This is question 6. Its hint is at the end of the notebook.

In [ ]:
# your turn: two variables named winner and log_gap
score_rrna = -1206.8
score_nonrrna = -1612.5

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("winner", expect="rRNA",
      hint="the less negative score is the larger one: compare score_rrna > score_nonrrna")
check("log_gap", expect=405.7)

### If you finish part 2 early

- Chapter 5's `log_likelihood` is section 2.3 with one term per k-mer. Read it beside the chapter's `likelihood`, line against line, and name the one operation that changed.
- Predict whether `steps_to_zero(0.5)` takes more steps than `steps_to_zero(0.3)` or fewer, then run it.
- PS5 is open.

### If you are stuck in part 2

Wave someone over. This hour exists so that a stuck step costs you a minute
rather than an evening. Known snags:

- **`steps_to_zero` returns `None`.** The `if` must sit inside the loop, and it must compare the product with exactly `0.0`.
- **`log_gap` is negative.** Subtract the smaller score from the larger one.
- **`is_false_discard(0.9, "non-rRNA")` is False.** *At least 0.9* includes 0.9, so the comparison is `>=`.

## Part 3 · Reading drills

**In this part:** you can trace a loop over a dictionary of dictionaries by hand, and you can read chapter 5's two scoring functions, the product and the log sum, and say which class each one picks.

### 3.1 · Trace the two-level lookup

The dictionary below counts twenty-five labeled reads by class and by whether k-mer 0 is present. The counts are typed by hand in the shape of chapter 5's first twenty-five labeled reads, six of which are rRNA. **By hand, before running: what prints?** Track which inner dictionary each pass of the loop opens.

In [ ]:
counts = {"rRNA":     {"with_kmer_0": 4, "without_kmer_0": 2},
          "non-rRNA": {"with_kmer_0": 7, "without_kmer_0": 12}}
total = 0
for label in counts:
    table = counts[label]
    total += table["with_kmer_0"]
print(total)
print(len(counts), len(counts["rRNA"]))

The first line is 11. The loop visits both classes and adds only the `"with_kmer_0"` count from each, 4 and then 7. The second line is `2 2`, because `len` counts the keys at the level it is given and nothing inside them.

**Test your understanding.** Write a function named `grand_total` that takes one parameter, `counts`, a dictionary of dictionaries of whole numbers shaped like the one above. It returns one whole number, the sum of every inner value under every outer key. Use a loop inside a loop. This is question 7. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named grand_total(counts)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("grand_total", expect=25,
      args=({"rRNA": {"with_kmer_0": 4, "without_kmer_0": 2},
             "non-rRNA": {"with_kmer_0": 7, "without_kmer_0": 12}},))
check("grand_total", expect=0, args=({},),
      hint="an empty dictionary totals 0. Does your function return 0 for it?")

### 3.2 · Which scale is each comparison on?

The cell below compares two probabilities, then compares their logarithms. **Before running: is each printed comparison made on probabilities or on logs, and do the two agree?**

In [ ]:
p_first = 0.004
p_second = 0.0032
log_first = np.log(p_first)
log_second = np.log(p_second)
print("probability scale:", p_first > p_second)
print("log scale:        ", log_first > log_second)

Both lines print True. Logs keep order, so a comparison of logs picks the same winner as a comparison of the probabilities. Chapter 5 relies on this property when it compares log sums in place of products.

**Test your understanding.** Create a variable named `always_negative` that stores `True` or `False`, the answer to one question: is `np.log` of a probability strictly between 0 and 1 always negative? Try `np.log` on three or four probabilities in the cell first, and decide from what they print. This is question 8. Its hint is at the end of the notebook.

In [ ]:
# your turn: try np.log on a few probabilities, then a variable named always_negative

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("always_negative", expect=True,
      hint="log(1) is exactly 0, and a probability strictly between 0 and 1 has a log below 0: try 0.99, 0.5 and 0.01")

### 3.3 · The chapter's two scores, read cold

The cell below uses only what this lab has shown. `profile` is the read profile that chapter 5 types by hand in *Twenty yes-or-no answers per read*, twenty answers with 1 for present and 0 for absent. `likelihood` and `log_likelihood` are the chapter's two functions. The loop scores the profile under each class's table in `presence_prob`, from the setup cell. `if present[i]:` treats 1 as True and 0 as False. Read every line before you run it. **Predict: which class has the larger product? Does the same class have the larger log sum? Does either product reach 0.0?**

In [ ]:
profile = [0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1]

def likelihood(present, probs):
    """P(profile | class) under independence: one factor per k-mer."""
    likelihood_p = 1.0
    for i in range(len(probs)):
        if present[i]:
            likelihood_p = likelihood_p * probs[i]
        else:
            likelihood_p = likelihood_p * (1 - probs[i])
    return likelihood_p

def log_likelihood(present, probs):
    """The same score as a sum of logs: one term per k-mer."""
    total = 0.0
    for i in range(len(probs)):
        if present[i]:
            total = total + np.log(probs[i])
        else:
            total = total + np.log(1 - probs[i])
    return total

for label in presence_prob:
    probs = presence_prob[label]
    print(f"{label:>8}   product: {likelihood(profile, probs):.2e}   log sum: {log_likelihood(profile, probs):.2f}")

rRNA has the larger product, 6.38e-08 against 3.44e-08, and also the larger log sum, −16.57 against −17.19. Both products are small, because twenty factors below 1 were multiplied, and at twenty k-mers neither reaches 0.0. The log sums are ordinary negative numbers, and they rank the two classes in the same order as the products.

**Test your understanding.** Write a function named `log_call` that takes one parameter, `present`, a list of twenty answers, each 1 or 0. It returns one string, `"rRNA"` or `"non-rRNA"`, the class whose `log_likelihood` is larger. Inside the function, call `log_likelihood` from the cell above, once with `presence_prob["rRNA"]` and once with `presence_prob["non-rRNA"]`. This is question 9. Its hint is at the end of the notebook.

In [ ]:
# your turn: a function named log_call(present)

In [ ]:
# run, don't edit — self-check
from labcheck import check

check("log_call", expect="rRNA",
      args=([0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1],))
check("log_call", expect="non-rRNA",
      args=([1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0],),
      hint="this profile is the first one flipped, every 1 made 0 and every 0 made 1, and it scores higher under non-rRNA")

### If you finish part 3 early

- Change one answer in `profile` from 0 to 1, predict whether `log_call` changes its answer, then run it.
- Chapter 5's practice problem 5.1, *Predict the output*, asks the same kind of question as this part.
- PS5 is open.

### If you are stuck in part 3

Wave someone over. This hour exists so that a stuck step costs you a minute
rather than an evening. Known snags:

- **`grand_total` fails the empty check.** Start `total = 0` before either loop. A loop over an empty dictionary never runs, so the function returns 0 with no extra code.
- **You disagree with `always_negative`.** `np.log(0.99)` is about −0.01, which is negative. `np.log(1)` is exactly 0, and the task says *strictly* between 0 and 1 to leave 1 out.
- **`log_call` raises `NameError` for `log_likelihood`.** Run the cell in section 3.3 first. It defines `log_likelihood`, and your function calls it.

## Hints

**Question 1 · `total_reads`, `rrna_share`.** Start a variable at 0 and loop over the dictionary with `for key in reads_by_class:`. Inside the loop, add `reads_by_class[key]` to it, as section 1.1 looks values up. After the loop, store the total as `total_reads`. Then `rrna_share` is `reads_by_class["rRNA"]` divided by `total_reads`, passed to `round` with 4.

**Question 2 · `class_average`, `gap`.** Inside the function, name the inner dictionary first, `table = presence[class_name]`, as section 1.2 does. Loop over the keys of `table` and add up `table[key]`. Divide the total by `len(table)` and round to 2. `gap` is one line that calls the function twice, subtracts, and rounds.

**Question 3 · `count_labels`.** Section 1.3's cell is the function's body. Start `tally = {"rRNA": 0, "non-rRNA": 0}`, loop over `labels` adding 1 under each label, and return `tally` after the loop.

**Question 4 · `is_false_discard`.** One `return` line is enough. Join `posterior >= 0.9` and `truth == "non-rRNA"` with `and`, as section 2.1 joins two questions.

**Question 5 · `steps_to_zero`.** Start `product = 1.0`, then loop with `for step in range(1, 100001):`. Inside the loop, multiply `product` by `factor`, then write `if product == 0.0:` with `return step` under it. Lab 3 used the same early `return` from inside a loop.

**Question 6 · `winner`, `log_gap`.** Write `if score_rrna > score_nonrrna:` and set `winner = "rRNA"` and `log_gap = round(score_rrna - score_nonrrna, 1)` under it. The `else` branch does the opposite. Section 2.3 says why the larger log sum is the winner.

**Question 7 · `grand_total`.** Start `total = 0`. The outer loop visits the class names, as in section 3.1. Name the inner dictionary, then loop over its keys and add each value to `total`. Return `total` after both loops.

**Question 8 · `always_negative`.** Print `np.log(0.99)`, `np.log(0.5)` and `np.log(0.01)`, and look at their signs. Then set the variable to the answer your prints support.

**Question 9 · `log_call`.** Compute two log sums, `log_likelihood(present, presence_prob["rRNA"])` and the same with `"non-rRNA"`, as the loop in section 3.3 does. Then an `if`/`else` returns the name of the class with the larger sum, as in question 6.